# 레슨 01 — 최종 미션 정답 / 모범 답안

> 🔒 **교사·관리자 전용. 학생에게 배포 금지.**

이 노트북은 최종 미션의 모범 답안이다. 학생 답안 채점·강의 시연·코드 검토에 사용한다.

## 단계 1. 환경 셀

In [ ]:
import os
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
DATA_BASE = "./data"
print("data base:", DATA_BASE)

## 단계 2. 데이터 로드 + 검증

In [ ]:
steps_raw = np.loadtxt(f"{DATA_BASE}/daily_steps.csv", delimiter=",", skiprows=1)
temps_raw = np.loadtxt(f"{DATA_BASE}/temperatures.csv", delimiter=",", skiprows=1)

day_s = steps_raw[:, 0].astype(int)
day_t = temps_raw[:, 0].astype(int)
steps = steps_raw[:, 1]
temp = temps_raw[:, 1]

assert steps.size == temp.size == 90, "두 파일의 길이가 다릅니다"
assert np.array_equal(day_s, day_t), "두 파일의 day 가 일치하지 않습니다"
print("로드 완료:", steps.size, "일치")

## 단계 3. 전체 통계

In [ ]:
print(f"평균 걸음: {steps.mean():,.0f} 보 (표준편차 {steps.std():,.0f})")
print(f"평균 기온: {temp.mean():.2f} °C (표준편차 {temp.std():.2f})")
print(f"중앙값 걸음: {np.median(steps):,.0f} 보")

## 단계 4. 월별 비교

In [ ]:
m1 = steps[:30]
m2 = steps[30:60]
m3 = steps[60:90]

print(f"1개월(1~30일)  평균: {m1.mean():,.0f} 보")
print(f"2개월(31~60일) 평균: {m2.mean():,.0f} 보")
print(f"3개월(61~90일) 평균: {m3.mean():,.0f} 보")

# 단순 추세 판정
diffs = [m2.mean() - m1.mean(), m3.mean() - m2.mean()]
if all(d > 200 for d in diffs):
    print("→ 점진적 증가 추세")
elif all(d < -200 for d in diffs):
    print("→ 점진적 감소 추세")
else:
    print("→ 뚜렷한 추세 없음(횡보)")

## 단계 5. 기온 분기 비교

In [ ]:
avg_t = temp.mean()
cold_mask = temp < avg_t
warm_mask = ~cold_mask

cold_avg = steps[cold_mask].mean()
warm_avg = steps[warm_mask].mean()

print(f"평균 미만 기온 날 ({cold_mask.sum()}일) 평균 걸음: {cold_avg:,.0f} 보")
print(f"평균 이상 기온 날 ({warm_mask.sum()}일) 평균 걸음: {warm_avg:,.0f} 보")
print(f"차이: {warm_avg - cold_avg:+,.0f} 보 (양수면 더운 날 더 걸음)")

## 단계 6. 평일/주말 비교

In [ ]:
dow = (day_s - 1) % 7
weekend_mask = (dow == 5) | (dow == 6)
weekday_mask = ~weekend_mask

wd = steps[weekday_mask].mean()
we = steps[weekend_mask].mean()

print(f"평일 평균: {wd:,.0f} 보 ({weekday_mask.sum()}일)")
print(f"주말 평균: {we:,.0f} 보 ({weekend_mask.sum()}일)")
print(f"평일 - 주말: {wd - we:+,.0f} 보")

## 단계 7. 상관계수 (보너스 B1)

In [ ]:
corr = np.corrcoef(temp, steps)[0, 1]
print(f"기온-걸음 상관계수: {corr:+.3f}")

if corr > 0.3:
    print("→ 양의 상관: 따뜻한 날 더 많이 걷는 경향")
elif corr < -0.3:
    print("→ 음의 상관: 추운 날 더 많이 걷는 경향")
else:
    print("→ 약한 상관: 기온과 걸음 수는 큰 관련이 없음")

## 보너스 B2 — 주별 평균과 최저 주차

In [ ]:
# 90일 / 7 = 12 주 + 6일. 마지막 주는 짧다.
weekly_means = []
for w in range(13):
    block = steps[w*7 : (w+1)*7]
    if block.size > 0:
        weekly_means.append(block.mean())
weekly_arr = np.array(weekly_means)

worst_week = int(weekly_arr.argmin())
print(f"가장 부족했던 주: {worst_week + 1}주차 (평균 {weekly_arr[worst_week]:,.0f}보)")
print(f"전체 주별 평균: {weekly_arr.astype(int).tolist()}")

## 보너스 B3 — 10,000보 연속 달성 일수

In [ ]:
# NumPy 로 연속 길이 구하기: True/False 의 변경점을 잡는다
goal_mask = (steps >= 10000).astype(int)
# 차이가 1 인 곳이 False→True (시작), -1 인 곳이 True→False (끝)
padded = np.concatenate([[0], goal_mask, [0]])
diffs = np.diff(padded)
starts = np.where(diffs == 1)[0]
ends = np.where(diffs == -1)[0]
runs = ends - starts
if runs.size > 0:
    max_run = int(runs.max())
    print(f"10,000보 연속 최장 기록: {max_run}일")
else:
    print("10,000보 달성한 날이 없음")

## 단계 8. 결론 예시 (모범 답)

In [ ]:
%%markdown
## 결론

1. **월별 추세**: 1개월 평균 8,200보 → 2개월 8,650보 → 3개월 8,400보로 큰 변화 없이 횡보했다. 폭발적 증가도, 명확한 이탈도 없다.
2. **기온 영향**: 평균 기온 이상인 날의 일평균이 평균 미만인 날보다 약 1,000보 많다. 약하지만 따뜻한 날 더 걷는 경향이 보이며 상관계수도 +0.21 로 같은 방향이다.
3. **평일/주말**: 평일 평균이 주말보다 약 3,800보 많다. 주말 활동이 명확히 부족하다.
4. **캠페인 톤**: "주말도 7,000보, 어렵지 않아요" 처럼 주말 부족 보충을 가볍게 응원하는 메시지가 효과적이다. 무리한 목표 제시(예: 매일 12,000보)보다는 평일/주말 격차를 줄이는 방향.

채점 시 결론 문장은 코드 결과 수치와 일치해야 한다. 데이터를 본 적 없는 듯한 결론(허구의 숫자, 데이터에 없는 변수 언급)은 감점.

## 학생 답안에서 자주 보는 패턴

| 패턴 | 의미 | 코멘트 |
|---|---|---|
| `steps[:30].mean()` 식으로 한 줄 처리 | 빠르고 정확 | 칭찬할 가치 있음 |
| 분기 마스크가 합집합이 안 됨 (`<` 와 `>` 만 사용) | 평균값 같은 날 누락 | 지적 + 다시 시도 |
| 결론에 수치 없음 | 데이터 근거 부족 | 부분 감점 |
| 상관계수 음수인데 양의 상관이라고 해석 | 부호 오독 | 보너스 점수 미부여 |
| 보너스 B3 를 for 루프 + 카운터로 풀이 | 정답이지만 비효율 | 통과시키되 NumPy 방식 시연으로 보여줌 |
| `corrcoef` 결과를 그대로 출력 (2x2 행렬) | 학생이 `[0,1]` 모름 | 살짝 도와주고 통과 |

## 채점 후 강사 메모

- 학생 결론이 "팀장 질문 4개에 정확히 한 줄씩" 짜였는지가 가장 중요. 줄 수가 안 맞으면 다시 작성하게 한다.
- 보너스 B3 는 1번 레슨 범위를 넘어가는 어려움(connected components 비슷). 못 풀어도 정상이다. 5명 중 1명이 풀면 잘하는 반.
- 학생이 데이터 안 받은 채로 결론만 그럴듯하게 쓴 경우(LLM 의존), 셀 출력값과 결론 수치가 어긋난다. 그 점이 가장 큰 적발 단서.